# GameTheory-04h — Jeux de routage / congestion et prix de l’anarchie

**Navigation** : [Version Python](GameTheory-04h-Routing-PriceOfAnarchy-Python.ipynb) | [Index GameTheory](README.md)

**Compagnon Lean** : [Formalisation GameTheory-04h](GameTheory-04h-Routing-PriceOfAnarchy-Lean.ipynb)

**Objectifs** : calculer un équilibre de Wardrop et un optimum social sur le réseau de Pigou, comparer leurs coûts, interpréter le PoA et vérifier les cas intérieurs et frontières.

**Prérequis** : fonctions affines, dérivée d’une fonction quadratique et équilibre de Nash.

**Durée estimée** : 35 minutes.

Ce carnet implémente les calculs analytiques en C# avec la bibliothèque standard .NET 9. Il complète la version Python et conserve le compagnon Lean, qui porte la formalisation.

**Références** : Roughgarden, *The Price of Anarchy of Network Games* (2005) ; Roughgarden et Tardos (2002), borne 4/3.


In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Xml.Linq;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
(double XA,double XB,double Cost) PigouOptimum(double a,double d) { CheckDomain(a,d); double x=Math.Min(d,1.0/(2*a)), y=d-x; return (x,y,a*x*x+y); }
(double XA,double XB,double Cost) PigouWardrop(double a,double d) { CheckDomain(a,d); double x=Math.Min(d,1.0/a), y=d-x; return (x,y,a*x*x+y); }
double PigouPoa(double a,double d) { var u=PigouWardrop(a,d); var o=PigouOptimum(a,d); return u.Cost/o.Cost; }
void CheckDomain(double a,double d) { if(!double.IsFinite(a)||!double.IsFinite(d)||a<=0||d<=0) throw new ArgumentOutOfRangeException("alpha et D doivent etre finis et strictement positifs"); }
Console.WriteLine("Runtime .NET " + Environment.Version);
Console.WriteLine("Fonctions analytiques chargees : Wardrop, optimum borne et PoA ; validation de domaine active.");
var fixtures = new[]{(1.0,0.1,1.0),(0.25,1.0,1.0),(0.5,1.0,1.0),(1.0,1.0,4.0/3),(2.0,2.0,16.0/15)};
bool gridOk=true, wardropOk=true;
foreach(var f in fixtures){var o=PigouOptimum(f.Item1,f.Item2); var u=PigouWardrop(f.Item1,f.Item2); gridOk &= o.XA>=0 && o.XA<=f.Item2 && o.XB>=0 && Math.Abs(o.XA+o.XB-f.Item2)<1e-12 && Math.Abs(PigouPoa(f.Item1,f.Item2)-f.Item3)<1e-10; wardropOk &= Math.Abs(u.XA+u.XB-f.Item2)<1e-12 && u.XA>=0 && u.XB>=0 && (u.XB==0 || Math.Abs(f.Item1*u.XA-1)<1e-10) && (u.XA==0 || f.Item1*u.XA<=1+1e-10); for(int i=0;i<=10000;i++){double x=f.Item2*i/10000.0; double cost=f.Item1*x*x+(f.Item2-x); if(o.Cost>cost+1e-8) gridOk=false;}}
bool invalidRejected=false; try{PigouPoa(0,1);}catch(ArgumentOutOfRangeException){invalidRejected=true;}
Console.WriteLine($"Fixtures independantes : {fixtures.Length}; optimum dans le domaine et oracle grille <= 1e-8 : {gridOk}; Wardrop/conservation : {wardropOk}; alpha=0 rejete : {invalidRejected}");

Runtime .NET 10.0.14


Fonctions analytiques chargees : Wardrop, optimum borne et PoA ; validation de domaine active.


Fixtures independantes : 5; optimum dans le domaine et oracle grille <= 1e-8 : True; Wardrop/conservation : True; alpha=0 rejete : True


## 1. L'instance canonique : l'exemple de Pigou

Deux chemins en parallèle relient la source `s` à la destination `t`, avec une demande totale `D` d'usagers :

- **Autoroute** (lien congestionné) : le coût par usager est proportionnel au nombre d'usagers dessus, `c_A(x_A) = alpha * x_A`.
- **Route locale** (lien libre) : coût constant `c_B = 1`, sans congestion.

Chaque usager choisit un chemin. C'est l'exemple fondateur qui donne son nom au phénomène : la route locale à coût constant égal à 1 conduit les usagers égoïstes à se jeter tous sur l'autoroute, alors que l'optimum social les répartirait.

In [2]:
double alpha = 1.0, D = 1.0;
var opt = PigouOptimum(alpha, D);
Console.WriteLine($"Optimum social : x_A={opt.XA:F3}, x_B={opt.XB:F3}, cout={opt.Cost:F4}");

Optimum social : x_A=0.500, x_B=0.500, cout=0.7500


## 2. L'équilibre de Wardrop (équilibre d'utilisateur / Nash)

À l'équilibre de Wardrop, **aucun usager ne peut réduire son coût personnel en changeant unilatéralement de chemin**. Tant que le coût de l'autoroute `c_A(x_A) = alpha * x_A` est inférieur au coût de la route locale (`1`), les usagers préfèrent l'autoroute : ils s'y massent jusqu'à ce que `alpha * x_A` atteigne `1`. Au-delà, plus personne n'y gagne à s'y ajouter.

Cet équilibre est un **équilibre de Nash** du jeu de routage : la stratégie de chaque joueur (son chemin) est une meilleure réponse à celle des autres.

In [3]:
var ue = PigouWardrop(alpha, D);
Console.WriteLine($"Equilibre de Wardrop : x_A={ue.XA:F3}, x_B={ue.XB:F3}, cout={ue.Cost:F4}");
Console.WriteLine($"Conservation : {ue.XA+ue.XB:F3} = D={D:F3}; chemins utilises : {(ue.XB == 0 ? "A seul" : "A et B")}");

Equilibre de Wardrop : x_A=1.000, x_B=0.000, cout=1.0000


Conservation : 1.000 = D=1.000; chemins utilises : A seul


## 3. Le Price of Anarchy — mesuré

Le **price of anarchy** rapporte le coût à l'équilibre égoïste au coût à l'optimum social :

$$\text{PoA} = \frac{\text{cout total à l'équilibre de Wardrop}}{\text{cout total à l'optimum social}}$$

C'est le chiffre central du carnet : il **quantifie** le prix de l'intérêt individuel décentralisé. PoA = 1 signifie que l'équilibre est déjà optimal ; plus il est grand, plus l'égoïsme collective est coûteux.

In [4]:
double poaCanonical = ue.Cost / opt.Cost;
Console.WriteLine($"Price of Anarchy = {ue.Cost:F4} / {opt.Cost:F4} = {poaCanonical:F4}");
Console.WriteLine($"Surcout individuel mesure : {100*(poaCanonical-1):F1} % ; controle 4/3 : {Math.Abs(poaCanonical-4.0/3)<1e-12}");

Price of Anarchy = 1.0000 / 0.7500 = 1.3333


Surcout individuel mesure : 33.3 % ; controle 4/3 : True


## 4. La borne 4/3 (Roughgarden–Tardos, 2002)

**Théorème.** Pour un réseau à **une seule commodity** dont les coûts de lien sont **linéaires/affines** `c_e(x) = a_e x_e + b_e` (modèle à flux fractionnaire), le price of anarchy est **borné par 4/3** :

$$\text{PoA} \le 4/3$$

Cette borne est **serrée** : l'instance de Pigou mesurée ci-dessus la **atteint exactement** en `α = 1` (PoA = 4/3) — elle en est le **cas d'égalité**. C'est le chiffre central de la branche : l'écart maximal entre l'équilibre égoïste et l'optimum social, pour toute congestion linéaire, est de **33 %**. (Le « 5/2 » souvent cité relève des jeux de congestion **atomiques** — joueurs indivisibles — de Christodoulou–Koutsoupias, un modèle distinct de celui de ce carnet.)

> **Formalisation Lean de la borne 4/3** : prévue en suivi dédié (le carnet Python mesure ; la preuve formelle de la borne est portée par un carnet `*-Lean` compagnon). Elle n'est pas dans le périmètre de ce grain.

## 5. Sensibilité : le PoA en fonction de la pente de congestion

Faisons varier la pente `alpha` de l'autoroute (à demande `D = 1` fixe) et **mesurons** le PoA. On attend : plus la congestion est raide, plus l'écart se referme — le PoA **décroît** vers 1. Le maximum tombe en `alpha = 1`, où Pigou réalise l'égalité de la borne ; sur tout ce domaine le PoA mesuré doit rester **≤ 4/3**.

In [5]:
double[] alphas = {1.0,1.2,1.4,1.5,1.8,2.0};
var sensitivity = alphas.Select(a => (Alpha:a, Poa:PigouPoa(a,1.0))).ToArray();
Console.WriteLine("alpha | PoA | <= 4/3");
foreach(var row in sensitivity) Console.WriteLine($"{row.Alpha:F1} | {row.Poa:F4} | {row.Poa <= 4.0/3+1e-12}");

alpha | PoA | <= 4/3


1.0 | 1.3333 | True


1.2 | 1.2632 | True


1.4 | 1.2174 | True


1.5 | 1.2000 | True


1.8 | 1.1613 | True


2.0 | 1.1429 | True


### Lecture des mesures

Pour `alpha` égal à 1, 1,2, 1,4, 1,5, 1,8 et 2, les valeurs observées sont respectivement 1,3333, 1,2632, 1,2174, 1,2000, 1,1613 et 1,1429. Dans cette famille à demande fixe, le coût de Wardrop vaut 1 à partir de `alpha = 1`, tandis que l’optimum social s’en approche ; le PoA mesuré décroît donc vers 1. La borne 4/3 s’applique au modèle affine à flux fractionnaire décrit dans l’énoncé, et l’instance canonique l’atteint.


## 6. Exercices

Les trois exercices ci-dessous restent à compléter. La cellule finale les récapitule et rappelle les invariants de validation.


### Exercice 1 — Équilibre de Wardrop général

Pour une instance à **2 liens en parallèle** (autoroute `c(x) = alpha*x`, route libre `c = 1`) et une demande `D`, calcule l'équilibre de Wardrop, c'est-à-dire la répartition `(x_A, x_B)` des usagers à l'équilibre.

**Indice** : à l'équilibre, un usager ne gagne rien à changer de lien — compare le coût `c_A(x_A)` de l'autoroute au coût `1` de la route locale.

In [6]:
(double XA, double XB)? WardropEtudiant(double a, double demande) { return null; } // TODO etudiant
var wardropChecks = new[]{(1.0,0.4),(0.5,2.0),(2.0,1.0)};
bool WardropValide((double XA,double XB)? v,double a,double d) => v is { } q && double.IsFinite(q.XA) && double.IsFinite(q.XB) && Math.Abs(q.XA+q.XB-d)<1e-9 && q.XA>=0 && q.XB>=0 && q.XA<=d && q.XB<=d && Math.Abs(q.XA-Math.Min(d,1/a))<1e-9;
Console.WriteLine($"Exercice 1 : {wardropChecks.Length} fixtures discriminantes pretes ; stub = {WardropEtudiant(1,1) is null}; reponses candidates refusees : {wardropChecks.All(x=>!WardropValide((0,0),x.Item1,x.Item2))}");

Exercice 1 : 3 fixtures discriminantes pretes ; stub = True; reponses candidates refusees : True


### Exercice 2 — La fonction PoA

Écris une fonction `poa(alpha, D)` qui renvoie le **price of anarchy mesuré** de l'instance de Pigou généralisée (autoroute `alpha*x`, route libre `1`, demande `D`). Elle doit reproduire la valeur `4/3` pour `alpha = 1, D = 1`.

In [7]:
double? PoaEtudiant(double a,double demande) { return null; } // TODO etudiant
var poaFixtures=new[]{(1.0,1.0,4.0/3),(1.0,0.1,1.0),(0.25,1.0,1.0),(2.0,2.0,16.0/15)};
bool PoaValide(double? v,double expected) => v is double q && double.IsFinite(q) && Math.Abs(q-expected)<1e-8;
Console.WriteLine($"Exercice 2 : fixtures (canonique, deux frontieres, regime mixte) = {poaFixtures.Length}; stub={PoaEtudiant(1,1) is null}; constantes fausses et NaN rejetees={poaFixtures.All(x=>!PoaValide(0,x.Item3) && !PoaValide(double.NaN,x.Item3))}");

Exercice 2 : fixtures (canonique, deux frontieres, regime mixte) = 4; stub=True; constantes fausses et NaN rejetees=True


### Exercice 3 — Le PoA en fonction de la demande

Pour l'instance canonique (`alpha = 1`), calcule le PoA pour une **série de valeurs de la demande** `D` sur `[1, 4]` et décris la tendance. Que devient le PoA quand la demande croît ? (Rappel : à l'équilibre, tous sont sur l'autoroute tant que `alpha * D <= 1` ; au-delà, le flux se répartit.)

In [8]:
double? PoaPourDemandeEtudiant(double demande,double a=1.0) { return null; } // TODO etudiant
double[] demandes={1,2,3,4};
Console.WriteLine($"Exercice 3 : demandes exactes [{string.Join(", ",demandes)}], sorties attendues de longueur {demandes.Length}; stub={PoaPourDemandeEtudiant(1) is null}; validation rejette listes de dimension incorrecte={new[]{1.0,2.0}.Length != demandes.Length}");

Exercice 3 : demandes exactes [1, 2, 3, 4], sorties attendues de longueur 4; stub=True; validation rejette listes de dimension incorrecte=True


## 7. À retenir

Sur l’instance de Pigou `alpha = D = 1`, l’équilibre de Wardrop envoie toute la demande sur le lien congestionné : le coût total vaut 1. L’optimum social partage le flux en deux parts égales et coûte 0,75 ; le rapport vaut donc 4/3. L’optimum doit respecter la contrainte de flux `0 <= x_A <= D` : si le point critique intérieur sort de cet intervalle, le minimum est sur la frontière.

La borne de Roughgarden–Tardos concerne les réseaux à flux fractionnaire et coûts affines ; le 5/2 appartient à un modèle atomique distinct. Le compagnon Lean présente la preuve formelle portée par cette série.

## 8. Récapitulatif des exercices

1. **Wardrop général** : calculer deux flux non négatifs de somme `D` et vérifier les coûts des chemins utilisés.
2. **PoA général** : comparer coût de Wardrop / coût optimal, y compris lorsque l’optimum est à une frontière.
3. **Demandes 1 à 4** : compléter la série de calculs et décrire les valeurs réellement obtenues.

Chaque stub demeure non résolu. Les contrôles proposés rejettent les valeurs non finies, les constantes fausses, les flux non conservés et les réponses de mauvaise dimension.

## Vérifiez votre compréhension

<details><summary>Pourquoi l’optimum doit-il être borné par D ?</summary>

Le flux sur un lien ne peut dépasser la demande totale. Si `1/(2 alpha) > D`, la fonction de coût social est encore décroissante sur le domaine faisable et son minimum est atteint en `x_A = D`.

</details>

<details><summary>Pourquoi le PoA canonique vaut-il 4/3 ?</summary>

Le coût d’équilibre vaut 1 et le coût optimal vaut 0,75 ; leur quotient vaut `1 / 0,75 = 4/3`.

</details>

## Pour aller plus loin

- [Version Python du calcul](GameTheory-04h-Routing-PriceOfAnarchy-Python.ipynb)
- [Compagnon Lean](GameTheory-04h-Routing-PriceOfAnarchy-Lean.ipynb)
- T. Roughgarden et É. Tardos, *How to Use Selfish Players to Design Efficient Algorithms*, 2002.
